In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt


In [ ]:

# ==========================================
# 1. Configuration
# ==========================================
# Point this to the folder containing your YOLO .txt annotation files
LABELS_DIR = './rdd2022_dataset/train/labels' 

# RDD2022 Class Mapping
CLASS_MAP = {
    0: 'D00_Longitudinal_Crack',
    1: 'D10_Transverse_Crack',
    2: 'D20_Alligator_Crack',
    3: 'D40_Pothole'
}



In [ ]:
# ==========================================
# 2. Data Extraction
# ==========================================
def extract_yolo_data(labels_dir):
    """Parses YOLO txt files into a structured Pandas DataFrame."""
    data = []
    
    # Grab all .txt files in the directory
    txt_files = glob.glob(os.path.join(labels_dir, '*.txt'))
    
    if not txt_files:
        print(f"No .txt files found in {labels_dir}. Please check your path.")
        return pd.DataFrame()

    for file_path in txt_files:
        filename = os.path.basename(file_path)
        
        # Extract country from filename (assuming format like "Japan_001.txt" or "Japan_001.jpg.txt")
        # Adjust the split logic based on your exact filename format
        country = filename.split('_')[0] 
        
        with open(file_path, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id = int(parts[0])
                # YOLO format: class_id, x_center, y_center, width, height (all normalized 0-1)
                width = float(parts[3])
                height = float(parts[4])
                
                # Filter to only keep the 4 main classes
                if class_id in CLASS_MAP:
                    data.append({
                        'filename': filename,
                        'country': country,
                        'class_id': class_id,
                        'class_name': CLASS_MAP[class_id],
                        'width': width,
                        'height': height,
                        # Area = width * height (since they are normalized, this represents % of image)
                        'area': width * height,
                        # Aspect Ratio = width / height
                        'aspect_ratio': width / height if height > 0 else 0 
                    })
                    
    return pd.DataFrame(data)



In [ ]:
# ==========================================
# 3. Statistical Analysis
# ==========================================
def run_statistical_tests(df):
    
    print("="*50)
    print("DATASET OVERVIEW")
    print("="*50)
    print(f"Total annotations found: {len(df)}")
    print(df['class_name'].value_counts())
    print("\n" + "="*50)

    # ---------------------------------------------------------
    # TEST 1: Chi-Square Test of Independence
    # Question: Is damage type dependent on the country/region?
    # ---------------------------------------------------------
    print("TEST 1: Chi-Square Test of Independence")
    print("H0: Damage type distribution is independent of country.")
    
    # Create a contingency table (cross-tabulation)
    contingency_table = pd.crosstab(df['country'], df['class_name'])
    print("\nContingency Table:\n", contingency_table)
    
    chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)
    print(f"\nChi-Square Statistic: {chi2:.4f}")
    print(f"P-value: {p_val:.4e}")
    
    if p_val < 0.05:
        print("Conclusion: Reject H0. The type of road damage is significantly dependent on the region.")
    else:
        print("Conclusion: Fail to reject H0. No significant dependency found.")
    print("="*50)

    # ---------------------------------------------------------
    # TEST 2: One-Way ANOVA
    # Question: Do different damage types have significantly different sizes (areas)?
    # ---------------------------------------------------------
    print("\nTEST 2: One-Way ANOVA (Bounding Box Area)")
    print("H0: All damage types have the same mean bounding box area.")
    
    # Group data by class
    groups = [df[df['class_id'] == class_id]['area'].values for class_id in CLASS_MAP.keys()]
    
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"\nF-Statistic: {f_stat:.4f}")
    print(f"P-value: {p_val:.4e}")
    
    if p_val < 0.05:
        print("Conclusion: Reject H0. At least one damage type is significantly different in size.")
    else:
        print("Conclusion: Fail to reject H0. No significant difference in mean area across types.")
    print("="*50)

    # ---------------------------------------------------------
    # TEST 3: Two-Sample Independent t-test
    # Question: Can we distinguish Longitudinal (D00) vs Transverse (D10) by aspect ratio?
    # ---------------------------------------------------------
    print("\nTEST 3: Independent t-test (Aspect Ratio)")
    print("H0: Longitudinal and Transverse cracks have the same mean aspect ratio.")
    
    d00_aspect = df[df['class_id'] == 0]['aspect_ratio']
    d10_aspect = df[df['class_id'] == 1]['aspect_ratio']
    
    t_stat, p_val = stats.ttest_ind(d00_aspect, d10_aspect, equal_var=False) # Welch's t-test
    
    print(f"\nT-Statistic: {t_stat:.4f}")
    print(f"P-value: {p_val:.4e}")
    
    if p_val < 0.05:
        print("Conclusion: Reject H0. Longitudinal and Transverse cracks have significantly different shapes.")
    else:
        print("Conclusion: Fail to reject H0. Shape alone may not distinguish these crack types.")
    print("="*50)



In [ ]:
# ==========================================
# 4. Execution
# ==========================================
if __name__ == "__main__":
    print("Loading data...")
    df = extract_yolo_data(LABELS_DIR)
    
    if not df.empty:
        run_statistical_tests(df)